# Loss landscape minima

How often does gradient descent from a random start find the reference? Starting from the
reference in `1104_loss_landscape_2d.ipynb` (config 7 with `tongueDiameter = 3`), draw many
uniformly random starting points for the trainable parameters, run a few optimizer steps
directly on the parameters, and look at the distribution of final losses. Then listen to runs at
every 20th percentile of the final loss.

Three arms, all from the same random starts with the same number of Adam steps:
- **plain**: gradient descent on the (range-normalized) parameter values.
- **gumbel (peaked)**: like training. Per-run logits over the 32 bucket centers, soft
  Gumbel-softmax at tau = 2, the synth gets the weighted mean of the bucket centers, plus
  training's hinged entropy penalty. The logits start as a peak a few buckets wide around the
  same random start as the plain run. Evaluated like training's eval: snapped to the argmax
  bucket.
- **gumbel (uniform)**: the same, but the logits start close to uniform (as in training, where
  the head's bias starts at zero), so every run starts near mid-range.
- **noisy**: control for gumbel. Plain gradient descent, but each forward pass evaluates the loss
  at the parameters plus Gaussian noise whose size matches the Gumbel scatter at the peaked init.
  If this does as well as gumbel, the benefit is the noise, not the bucket parametrization.

`frequency` is held at the reference value: in training it comes from pitch tracking, not from
the model. The loss is the middle-frame MFCC loss of a short clip, as in the 2D notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
from IPython.display import Audio, Markdown, display

from samuel.losses import MFCCLoss
from samuel.model import _DEFAULT_FROZEN_VALUES, _DEFAULT_PARAM_SPEC, PinkTromboneControllerConfig
from samuel.pink_trombone import PARAM_NAMES, SAMPLE_RATE, pink_trombone_ola

SAMPLES_PER_FRAME = 512
CONTROL_RATE = SAMPLE_RATE / SAMPLES_PER_FRAME
# The loss clip is 4096 samples -> 5 MFCC frames (n_fft 2048, hop 512); the middle one covers
# samples 1024-3072. Parameters are constant, so the clip is synthesized as a single 4096-sample
# control frame: one impulse response per run instead of 8, with the same audio.
LOSS_CLIP_SAMPLES = 4096
SEED = 0
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

N_RUNS = 100
N_STEPS = 150
LR = 0.02  # plain/noisy: in units of each parameter's full range

# Gumbel arm, matching configs/config.yaml.
N_BUCKETS = PinkTromboneControllerConfig().n_buckets  # 32
TAU = 2.0
ENTROPY_WEIGHT = 0.1
ENTROPY_FLOOR = 1.0  # nats
# Not matched to LR: Adam moves each logit by ~LR_LOGITS per step. Compare the distance
# travelled per arm below.
LR_LOGITS = 1.0
# Std of the initial logit peak, in units of the range: ~3 buckets for the peaked arm, and the
# whole range for the near-uniform one.
INIT_WIDTH_PEAKED = 3.0 / (N_BUCKETS - 1)
INIT_WIDTH_UNIFORM = 1.0

# The loss stays on CPU: torch.stft on MPS crashed the kernel. It's cheap next to the synth.
loss_fn = MFCCLoss(samples_per_frame=SAMPLES_PER_FRAME, n_fft=2048)

## Reference configuration

In [ ]:
config = {
    **_DEFAULT_FROZEN_VALUES,
    "voiceness": 0.64,
    "intensity": 0.74,
    "tongueIndex": 30.49,
    "tongueDiameter": 3.0,
    "constrictionIndex": 33.87,
    "constrictionDiameter": 2.01,
    "lipDiameter": 2.62,
    "frequency": 108.78,
}
TRAINABLE = list(_DEFAULT_PARAM_SPEC)
lo = torch.tensor([_DEFAULT_PARAM_SPEC[n][0] for n in TRAINABLE])
hi = torch.tensor([_DEFAULT_PARAM_SPEC[n][1] for n in TRAINABLE])
TRAINABLE

In [ ]:
def make_params(u: torch.Tensor, n_frames: int = 1) -> torch.Tensor:
    """[B, n_trainable] in [0, 1] (scaled to each range) -> [B, n_frames, 12], constant over time."""
    base = torch.tensor([config[n] for n in PARAM_NAMES], device=DEVICE)
    params = base.expand(len(u), n_frames, -1).clone()
    values = (lo + u * (hi - lo)).to(DEVICE)
    for j, name in enumerate(TRAINABLE):
        params[..., PARAM_NAMES.index(name)] = values[:, j, None]
    return params


def synth(params: torch.Tensor, control_rate: float = SAMPLE_RATE / LOSS_CLIP_SAMPLES) -> torch.Tensor:
    # A [B] tensor of equal seeds gives every row the same noise; a scalar seed
    # would draw a different noise row per batch slot.
    seed = torch.full((len(params),), SEED)
    return pink_trombone_ola(params, seed=seed, control_rate=control_rate).cpu()


def listen(u: torch.Tensor) -> Audio:
    """1 s of audio for one [1, n_trainable] row."""
    with torch.no_grad():
        audio = synth(make_params(u, n_frames=round(CONTROL_RATE)), control_rate=CONTROL_RATE)
    return Audio(audio[0].numpy(), rate=SAMPLE_RATE)


def middle_frame_features(audio: torch.Tensor) -> torch.Tensor:
    """[B, S] -> [B, n_mfcc] MFCCs of the middle frame."""
    feats = loss_fn.features(audio)  # [B, n_mfcc, T]
    return feats[..., feats.shape[-1] // 2]


u_ref = (torch.tensor([config[n] for n in TRAINABLE]) - lo) / (hi - lo)
with torch.no_grad():
    ref_features = middle_frame_features(synth(make_params(u_ref[None])))


def loss_at(u: torch.Tensor) -> torch.Tensor:
    """[B, n_trainable] in [0, 1] -> [B] middle-frame MFCC loss against the reference."""
    return (middle_frame_features(synth(make_params(u))) - ref_features).abs().mean(dim=-1)


listen(u_ref[None])

## Gradient descent

All runs of an arm go in one batch. Each run's loss only depends on its own parameters, so the
gradient of the summed loss gives every run its own gradient. Plain and noisy parameters are
clamped back into their ranges after each step.

In [ ]:
u0 = torch.rand(N_RUNS, len(TRAINABLE), generator=torch.Generator().manual_seed(SEED))
centers = torch.linspace(0.0, 1.0, N_BUCKETS)  # bucket centers in normalized units



def init_logits(width: float) -> torch.Tensor:
    """Logits peaked at each run's start: a discretized Gaussian of std ``width``."""
    return -0.5 * ((centers - u0[..., None]) / width) ** 2  # [N_RUNS, n_trainable, N_BUCKETS]



def gumbel_value(logits: torch.Tensor) -> torch.Tensor:
    return (F.gumbel_softmax(logits, tau=TAU, hard=False, dim=-1) * centers).sum(-1)


# Noise for the control arm: the spread of the Gumbel value at init, averaged over runs and params.
torch.manual_seed(SEED)
NOISE_STD = (
    torch.stack([gumbel_value(init_logits(INIT_WIDTH_PEAKED)) for _ in range(64)])
    .std(dim=0).mean().item()
)
print(f"NOISE_STD = {NOISE_STD:.4f} (in units of each parameter's range)")

In [ ]:
def run_values(noise_std: float):
    """Plain (noise_std=0) or noisy GD on the parameter values -> (train losses, final u)."""
    torch.manual_seed(SEED)
    u = u0.clone().requires_grad_(True)
    opt = torch.optim.Adam([u], lr=LR)
    history = []
    for step in range(N_STEPS):
        u_eval = (u + noise_std * torch.randn_like(u)).clamp(0.0, 1.0) if noise_std else u
        loss = loss_at(u_eval)
        history.append(loss.detach())
        if step % 10 == 0:
            print(f"step {step:3d}: median {loss.median():.3f}, min {loss.min():.3f}")
        opt.zero_grad()
        loss.sum().backward()
        opt.step()
        with torch.no_grad():
            u.clamp_(0.0, 1.0)
    return torch.stack(history), u.detach()


def run_gumbel(init_width: float):
    """Soft Gumbel-softmax over bucket centers -> (train losses, final logits)."""
    torch.manual_seed(SEED)
    logits = init_logits(init_width).requires_grad_(True)
    opt = torch.optim.Adam([logits], lr=LR_LOGITS)
    history = []
    for step in range(N_STEPS):
        loss = loss_at(gumbel_value(logits))
        history.append(loss.detach())
        if step % 10 == 0:
            print(f"step {step:3d}: median {loss.median():.3f}, min {loss.min():.3f}")
        log_probs = logits.log_softmax(-1)
        entropy = -(log_probs.exp() * log_probs).sum(-1)  # [N_RUNS, n_trainable]
        penalty = F.relu(ENTROPY_FLOOR - entropy).mean(-1)  # [N_RUNS]
        opt.zero_grad()
        (loss + ENTROPY_WEIGHT * penalty).sum().backward()
        opt.step()
    return torch.stack(history), logits.detach()

In [ ]:
print("plain")
hist_plain, u_plain = run_values(noise_std=0.0)
print("noisy")
hist_noisy, u_noisy = run_values(noise_std=NOISE_STD)
print("gumbel (peaked)")
hist_peaked, logits_peaked = run_gumbel(INIT_WIDTH_PEAKED)
print("gumbel (uniform)")
hist_uniform, logits_uniform = run_gumbel(INIT_WIDTH_UNIFORM)

final_u = {"plain": u_plain, "noisy": u_noisy}
for name, logits in [("peaked", logits_peaked), ("uniform", logits_uniform)]:
    final_u[f"gumbel {name} (argmax)"] = centers[logits.argmax(-1)]
    final_u[f"gumbel {name} (expected)"] = (logits.softmax(-1) * centers).sum(-1)
with torch.no_grad():
    final_loss = {"start": loss_at(u0), **{name: loss_at(u) for name, u in final_u.items()}}

## Results

In [ ]:
THRESHOLD = 0.05
rows = []
for name, losses in final_loss.items():
    travelled = (final_u[name] - u0).abs().mean() if name in final_u else torch.tensor(0.0)
    rows.append(
        f"| {name} | {losses.median():.3f} | {(losses < THRESHOLD).float().mean():.0%} "
        f"| {travelled:.3f} |"
    )
display(Markdown(
    f"| arm | median final loss | runs below {THRESHOLD} | mean distance from start |\n"
    "|---|---|---|---|\n" + "\n".join(rows)
))

Results of the MPS run on 2026-11-04:

| arm | median final loss | runs below 0.05 | mean distance from start |
|---|---|---|---|
| start | 0.607 | 0% | 0.000 |
| plain | 0.058 | 47% | 0.177 |
| noisy | 0.122 | 17% | 0.197 |
| gumbel peaked (argmax) | 0.107 | 17% | 0.170 |
| gumbel peaked (expected) | 0.065 | 44% | 0.171 |
| gumbel uniform (argmax) | 0.085 | 20% | 0.286 |
| gumbel uniform (expected) | 0.041 | 69% | 0.291 |

In [ ]:
fig = go.Figure()
for name, losses in final_loss.items():
    fig.add_trace(go.Histogram(x=losses, name=name, nbinsx=40, opacity=0.5))
fig.update_layout(
    barmode="overlay", height=420, xaxis_title="MFCC loss (middle frame)", yaxis_title="runs",
    title=f"Final loss of {N_RUNS} random starts after {N_STEPS} steps",
)
fig.show()

In [ ]:
fig = go.Figure()
for name, hist in [
    ("plain", hist_plain), ("noisy", hist_noisy),
    ("gumbel (peaked)", hist_peaked), ("gumbel (uniform)", hist_uniform),
]:
    fig.add_trace(go.Scatter(y=hist.median(dim=1).values, name=name))
fig.update_layout(
    height=380, xaxis_title="step", yaxis_title="median training loss",
    title="Median training loss (noisy and gumbel are measured at the perturbed values)",
)
fig.show()

### Runs by percentile of final loss

For each arm, every 20th percentile from the best run (0th) to the worst (100th), rendered for
1 s with the final parameters.

In [ ]:
display(Markdown("**Reference:** " + ", ".join(f"{n}={config[n]:.2f}" for n in TRAINABLE)))
display(listen(u_ref[None]))
for arm in ["plain", "noisy", "gumbel peaked (argmax)", "gumbel uniform (argmax)"]:
    display(Markdown(f"#### {arm}"))
    final = final_loss[arm]
    order = final.argsort()
    for q in [0, 20, 40, 60, 80, 100]:
        i = order[round(q / 100 * (N_RUNS - 1))].item()
        u = final_u[arm][i : i + 1]
        values = (lo + u[0] * (hi - lo)).tolist()
        display(Markdown(
            f"**{q}th percentile** (run {i}, loss {final[i]:.3f}): "
            + ", ".join(f"{n}={v:.2f}" for n, v in zip(TRAINABLE, values))
        ))
        display(listen(u))